# 01 - Evaluate Detector
Evalueaza checkpoint-ul detector pe split-ul val/test si salveaza summary JSON.

In [ ]:
from pathlib import Path
import json
from ultralytics import YOLO

REPO_ROOT = Path('../..').resolve()
MODEL_PATH = REPO_ROOT / 'runs' / 'detect' / 'parks-trash-A22' / 'weights' / 'best.pt'
DATA_YAML = REPO_ROOT / 'datasets' / 'parks_detect' / 'parks_detect.yaml'
SPLIT = 'test'
IMGSZ = 416
BATCH = 8
CONF = 0.25
IOU = 0.6
DEVICE = '0'

print(MODEL_PATH)
print(DATA_YAML)

In [ ]:
assert MODEL_PATH.exists(), f'Missing checkpoint: {MODEL_PATH}'
assert DATA_YAML.exists(), f'Missing data yaml: {DATA_YAML}'

model = YOLO(str(MODEL_PATH))
results = model.val(
    data=str(DATA_YAML), split=SPLIT, imgsz=IMGSZ, batch=BATCH, conf=CONF, iou=IOU,
    device=DEVICE, plots=False, verbose=False
)

box = getattr(results, 'box', None)
if box is not None:
    print('P       :', float(box.mp))
    print('R       :', float(box.mr))
    print('mAP50   :', float(box.map50))
    print('mAP50-95:', float(box.map))

In [ ]:
summary = {
    'split': SPLIT,
    'model': str(MODEL_PATH),
    'data': str(DATA_YAML),
    'precision': float(getattr(box, 'mp', 0.0)) if box else None,
    'recall': float(getattr(box, 'mr', 0.0)) if box else None,
    'map50': float(getattr(box, 'map50', 0.0)) if box else None,
    'map50_95': float(getattr(box, 'map', 0.0)) if box else None,
}
out_dir = REPO_ROOT / 'runs' / 'detect_eval' / 'notebook_eval' / SPLIT
out_dir.mkdir(parents=True, exist_ok=True)
out_json = out_dir / 'summary.json'
out_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('saved:', out_json)
summary

# 01 — Evaluare Detector (Stage 1)

Evaluează un **detector YOLOv8 trash** pe split-ul `val` sau `test` și generează:
- Metrici: Precision, Recall, mAP@50, mAP@50-95
- Grafice: precision-recall curve, confusion matrix
- Vizualizare predicții pe imagini de test
- Fișier `summary.json` pentru rapoartele tezei

**Pre-condiție**: Ai rulat `notebooks/training/01_train_detector.ipynb`.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ultralytics import YOLO

print(f"Rădăcina proiectului: {REPO_ROOT}")

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────

# Calea către cel mai bun model antrenat
MODEL_PATH   = REPO_ROOT / "runs" / "detect" / "parks-trash-A22" / "weights" / "best.pt"

DATA_YAML    = str(REPO_ROOT / "datasets" / "parks_detect" / "parks_detect.yaml")
EVAL_SPLIT   = "test"       # "val" sau "test"
IMGSZ        = 416          # trebuie să coincidă cu imgsz din antrenare
BATCH        = 8
WORKERS      = 0            # 0 = stabil pe Windows
CONF         = 0.25
IOU          = 0.60
DEVICE       = "0"          # "0" = GPU RTX 3050 | "cpu"

OUTPUT_DIR   = REPO_ROOT / "runs" / "detect_eval" / MODEL_PATH.parent.parent.name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model    : {MODEL_PATH}")
print(f"Dataset  : {Path(DATA_YAML).name}   split={EVAL_SPLIT}")
print(f"Output   : {OUTPUT_DIR}")


---
## Evaluare metrici (mAP, P, R)

In [ ]:
assert MODEL_PATH.exists(), f"Modelul nu există: {MODEL_PATH}"

model = YOLO(str(MODEL_PATH))

val_kwargs = dict(
    data    = DATA_YAML,
    split   = EVAL_SPLIT,
    imgsz   = IMGSZ,
    batch   = BATCH,
    workers = WORKERS,
    conf    = CONF,
    iou     = IOU,
    device  = DEVICE,
    plots   = True,
    verbose = True,
)

results = model.val(**val_kwargs)

box = getattr(results, "box", None)


def to_python(v):
    return v.item() if hasattr(v, "item") else v


summary = {
    "model"    : str(MODEL_PATH),
    "dataset"  : DATA_YAML,
    "split"    : EVAL_SPLIT,
    "conf"     : CONF,
    "iou"      : IOU,
    "precision": to_python(box.mp)    if box else None,
    "recall"   : to_python(box.mr)    if box else None,
    "mAP50"    : to_python(box.map50) if box else None,
    "mAP5095"  : to_python(box.map)   if box else None,
    "metrics"  : {str(k): to_python(v) for k, v in getattr(results, "results_dict", {}).items()},
}

summary_path = OUTPUT_DIR / f"summary_{EVAL_SPLIT}.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("\n" + "="*50)
print(f"  Split      : {EVAL_SPLIT}")
print(f"  Precision  : {summary['precision']:.4f}" if summary['precision'] else "  Precision  : —")
print(f"  Recall     : {summary['recall']:.4f}"    if summary['recall']    else "  Recall     : —")
print(f"  mAP@50     : {summary['mAP50']:.4f}"     if summary['mAP50']     else "  mAP@50     : —")
print(f"  mAP@50-95  : {summary['mAP5095']:.4f}"   if summary['mAP5095']   else "  mAP@50-95  : —")
print("="*50)
print(f"\nSummary JSON salvat: {summary_path}")


---
## Grafice Ultralytics (PR curve, Confusion Matrix)
Ultralytics salvează automat grafice în directorul runs. Le afișăm direct.

In [ ]:
from IPython.display import Image as IPImage, display

save_dir = Path(getattr(results, "save_dir", str(OUTPUT_DIR)))
print(f"Grafice în: {save_dir}")

plots = [
    ("PR Curve",         "PR_curve.png"),
    ("Confusion Matrix", "confusion_matrix.png"),
    ("F1 Curve",         "F1_curve.png"),
]

for title, fname in plots:
    p = save_dir / fname
    if p.exists():
        print(f"\n--- {title} ---")
        display(IPImage(str(p), width=700))
    else:
        print(f"[INFO] {fname} nu a fost găsit")

---
## Vizualizare predicții pe imagini de test
Rulează inferența pe câteva imagini din split-ul de test și afișează bounding box-urile prezise.

In [ ]:
import yaml

N_PREVIEW    = 8       # număr de imagini de afișat
PREVIEW_CONF = 0.25
IMAGE_EXTS   = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Găsește directorul de test
with open(DATA_YAML) as f:
    ds = yaml.safe_load(f)
root = Path(ds["path"])
if not root.is_absolute():
    root = Path(DATA_YAML).parent / root

test_img_dir = root / ds.get("test", "images/test")
test_images  = sorted(p for p in test_img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

if not test_images:
    print(f"Nicio imagine de test în {test_img_dir}")
else:
    preview_imgs = test_images[:N_PREVIEW]
    fig, axes = plt.subplots(2, 4, figsize=(18, 9)) if len(preview_imgs) >= 4 else plt.subplots(1, len(preview_imgs), figsize=(5*len(preview_imgs), 5))
    axes = np.array(axes).flatten()

    for ax, img_path in zip(axes, preview_imgs):
        preds = model.predict(str(img_path), conf=PREVIEW_CONF, verbose=False)[0]
        frame = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ax.imshow(frame)
        for box in preds.boxes:
            x1, y1, x2, y2 = [int(c) for c in box.xyxy[0]]
            conf = float(box.conf[0])
            rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1-4, f"trash {conf:.2f}", color="lime", fontsize=8, fontweight="bold")
        ax.set_title(img_path.name[:30], fontsize=8)
        ax.axis("off")

    for ax in axes[len(preview_imgs):]:
        ax.axis("off")

    plt.suptitle(f"Predicții detector — split={EVAL_SPLIT}  conf≥{PREVIEW_CONF}", fontsize=12)
    plt.tight_layout()
    plt.show()

---
## Comparare experimente A1 / A2 / A3
Încarcă fișierele `summary_test.json` din fiecare run și afișează un bar chart comparativ.

In [ ]:
import glob

# Caută toate summary-urile disponibile
summaries_dir = REPO_ROOT / "runs" / "detect_eval"
summary_files = list(summaries_dir.glob("*/summary_test.json"))
if not summary_files:
    summary_files = list(summaries_dir.glob("*/summary_val.json"))

if not summary_files:
    print("Nu există fișiere summary JSON. Rulează celula de evaluare pentru fiecare experiment.")
else:
    exp_data = []
    for sf in sorted(summary_files):
        with open(sf) as f:
            s = json.load(f)
        run = Path(s["model"]).parent.parent.name
        exp_data.append({"run": run, "P": s.get("precision",0) or 0, "R": s.get("recall",0) or 0,
                         "mAP50": s.get("mAP50",0) or 0, "mAP5095": s.get("mAP5095",0) or 0})

    runs   = [d["run"]    for d in exp_data]
    p_vals = [d["P"]      for d in exp_data]
    r_vals = [d["R"]      for d in exp_data]
    m50    = [d["mAP50"]  for d in exp_data]
    m5095  = [d["mAP5095"] for d in exp_data]

    x      = np.arange(len(runs))
    width  = 0.18
    fig, ax = plt.subplots(figsize=(max(8, 2.5*len(runs)), 5))

    ax.bar(x - 1.5*width, p_vals, width, label="Precision",  color="steelblue")
    ax.bar(x - 0.5*width, r_vals, width, label="Recall",     color="tomato")
    ax.bar(x + 0.5*width, m50,   width, label="mAP@50",     color="mediumseagreen")
    ax.bar(x + 1.5*width, m5095, width, label="mAP@50-95",  color="darkorange")

    ax.set_xticks(x); ax.set_xticklabels(runs, rotation=15, ha="right")
    ax.set_ylim(0, 1.05); ax.set_ylabel("Valoare"); ax.set_title("Comparare Experimente Detector")
    ax.legend(loc="lower right")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Tabel
    print(f"\n{'Run':<25} {'P':>8} {'R':>8} {'mAP50':>9} {'mAP5095':>10}")
    print("-" * 65)
    for d in exp_data:
        print(f"{d['run']:<25} {d['P']:>8.4f} {d['R']:>8.4f} {d['mAP50']:>9.4f} {d['mAP5095']:>10.4f}")